# Context Managers
Context managers are a powerful feature in Python used to manage resources efficiently. They ensure that resources—like file streams, database connections, or locks—are properly and safely acquired and released, even if errors occur during execution.

## 1. The with Statement
The with statement is the standard way to utilize context managers in Python. It simplifies resource management by handling the setup and teardown phases automatically.

### How It Works Under the Hood
When you use a with statement, Python relies on two special methods defined in the target object (known as the Context Manager Protocol):

\_\_enter\_\_(): Executed before the block of code runs. It sets up the environment and can return a resource.

\_\_exit\_\_(): Executed after the block of code finishes (whether it succeeded or raised an exception). It handles cleanup tasks like closing files or releasing locks.

### Example: File Handling
Without a context manager, you have to manually remember to close files, which can lead to memory leaks or locked files if an exception crashes your program midway:

In [ ]:
# The Traditional (Risky) Way
f = open("data.txt", "w")
try:
    f.write("Hello, World!")
finally:
    f.close()  # Must remember to close manually

With the with statement, Python guarantees that f.close() is called automatically:

In [ ]:
# The Pythonic Way
with open("data.txt", "w") as f:
    f.write("Hello, World!")
# The file is automatically closed here, even if an exception occurs inside the block.

## 2. Custom Context Managers
You can create your own context managers to handle custom resources (like a database connection, a timer, or a temporary directory change). There are two primary ways to do this: Class-Based and Generator-Based.

### Approach A: Class-Based Context Managers
To create a custom context manager using a class, you simply implement the \_\_enter\_\_ and \_\_exit\_\_ magic methods.

In [ ]:
class ManagedFile:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        print(f"Opening file: {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file  # This is bound to the 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
            print(f"Closing file: {self.filename}")
        
        # Returning True suppresses any exception that occurred inside the 'with' block.
        # Returning None (or False) lets the exception propagate normally.
        return False

# Usage
with ManagedFile("notes.txt", "w") as f:
    f.write("Writing custom context manager notes.")

### Parameters of \_\_exit\_\_:

exc_type: The exception type (if an error occurred), otherwise None.

exc_val: The exception instance, otherwise None.

exc_tb: The traceback object, otherwise None.

## 3. The contextlib Module
Writing a full class with \_\_enter\_\_ and \_\_exit\_\_ can sometimes be verbose for simple tasks. Python's built-in contextlib module provides utilities to make writing context managers much easier.

### Using @contextmanager Decorator
You can turn a simple generator function into a context manager using the @contextmanager decorator.

Code before the yield statement acts as \_\_enter\_\_.

The value passed to yield becomes the variable assigned in the as clause.

Code after the yield statement acts as \_\_exit\_\_.

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    # Setup phase (__enter__)
    print(f"Acquiring resource: {name}")
    resource = {"name": name, "status": "active"}
    
    try:
        yield resource  # Hand control over to the 'with' block
    except Exception as e:
        print(f"An error occurred: {e}")
        raise  # Re-raise the exception after logging if needed
    finally:
        # Teardown phase (__exit__)
        print(f"Releasing resource: {name}")
        resource["status"] = "closed"

# Usage
with managed_resource("DatabaseConnection") as db:
    print(f"Working with {db['name']}")
    # Simulate an operation
    # raise ValueError("Connection lost!")

### Other Handy Tools in contextlib
contextlib.suppress(*exceptions): Temporarily suppresses specified exceptions.

In [ ]:
from contextlib import suppress

# Prevents FileNotFoundError from crashing the app if the file doesn't exist
with suppress(FileNotFoundError):
    import os
    os.remove("non_existent_file.txt")

contextlib.redirect_stdout(new_target): **Temporarily redirects** sys.stdout to another target (like a file or string buffer).

### Best Practices & Common Pitfalls
**Always use context managers for external resources:** Whenever your code interacts with files, network sockets, locks, or database transactions, wrap them in a context manager to prevent leaks.

**Don't swallow exceptions blindly in \_\_exit\_\_:** If you write a custom \_\_exit\_\_ method or handle exceptions inside a generator context manager, make sure you either intentionally suppress the exception (by returning True in a class or handling it cleanly) or let it propagate (raise). Swallowing exceptions silently makes debugging extremely difficult.

**Prefer contextlib for simplicity:** If your context manager only needs to handle basic setup and teardown logic without managing complex state transitions, use @contextmanager to keep your code concise and readable.